# Step 3-1 (A) : XGBoost Walk-Forward — Q, Ω 계산

> **목적**: ETF 22종 Long-Panel 데이터로 XGBoost 분류 모델을 Walk-Forward 방식으로 학습하고,
> Black-Litterman 입력값인 **Q** (기대수익률)와 **Ω** (뷰 불확실성)를 추출한다.

---

### Walk-Forward 구조 (Lopez de Prado, 2018)

```
│◀──── IS: 150일 ────▶│◀ Embargo: 21일 ▶│◀ OOS: 30일 ▶│  → 30일씩 슬라이딩
총 윈도우 수 ≈ (전체 날짜 - 150) / 30 ≈ 79개
```

### 출력 파일

| 파일 | 내용 |
|------|------|
| `outputs/step3/xgb_Q_Omega.parquet` | OOS 전체 Q, Ω, 확률, 실제수익률 |
| `outputs/step3/Q_xgb.parquet` | BL 입력용 Q 피벗 (date × ticker) |
| `outputs/step3/Omega_xgb.parquet` | BL 입력용 Ω 피벗 (date × ticker) |


In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import OrdinalEncoder
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path("data")
OUTPUT_DIR = Path("outputs/step3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Walk-Forward 파라미터 ──────────────────────────────────────────────────────
IS_DAYS         = 150   # 학습 구간 행 수 (일별 패널 기준 ≈ 7개월)
EMBARGO_DAYS    = 21    # IS-OOS 경계 엠바고 (Lopez de Prado)
OOS_DAYS        = 30    # 예측 구간 행 수
N_CLASSES       = 5     # 분류 등급: C1(강한하락) ~ C5(강한상승)
N_OPTUNA_TRIALS = 30    # Optuna 탐색 횟수 (속도 우선 시 10으로 줄여도 됨)
RANDOM_STATE    = 42

print("✅ 설정 완료")
print(f"  IS={IS_DAYS}일 | Embargo={EMBARGO_DAYS}일 | OOS={OOS_DAYS}일 | N_classes={N_CLASSES}")
print(f"  출력 경로: {OUTPUT_DIR.resolve()}")

✅ 설정 완료
  IS=150일 | Embargo=21일 | OOS=30일 | N_classes=5
  출력 경로: C:\workspace\camp\project\finance_project\서윤범\outputs\step3


## 2. 데이터 로드

Step2에서 저장된 `long_panel.parquet` 로드.  
Multi-Index: `[date, ticker]` / 타겟: `target_fwd` (21일 포워드 로그수익률)

In [4]:
# parquet 우선, 없으면 csv 시도
try:
    panel = pd.read_parquet(DATA_DIR / "long_panel.parquet")
    print("✅ parquet 로드 성공")
except FileNotFoundError:
    panel = pd.read_csv(
        DATA_DIR / "df_panel.csv",
        index_col=["date", "ticker"],
        parse_dates=["date"]
    )
    print("✅ csv 로드 성공")

panel = panel.sort_index()

print(f"\nPanel shape : {panel.shape}")
print(f"날짜 범위   : {panel.index.get_level_values('date').min().date()} "
      f"~ {panel.index.get_level_values('date').max().date()}")
print(f"ETF 수      : {panel.index.get_level_values('ticker').nunique()}")
print(f"고유 날짜 수: {panel.index.get_level_values('date').nunique()}")
print(f"\n컬럼 목록: {panel.columns.tolist()}")

ValueError: Missing column provided to 'parse_dates': 'date'

In [3]:
# 결측치 현황
missing = panel.isnull().mean().sort_values(ascending=False)
print("결측치 비율 (> 0인 컬럼만):")
print(missing[missing > 0].to_string())

NameError: name 'panel' is not defined

## 3. 피처 / 타겟 정의

In [ ]:
TARGET_COL = "target_fwd"   # 21일 포워드 로그수익률
SECTOR_COL = "sector_code"  # 카테고리 피처 (EQUITY/BOND/ALT)

# ── 자산별 피처 (모멘텀·변동성) ───────────────────────────────────────────────
ASSET_FEATURES = [
    "ret_1m", "ret_3m", "ret_6m", "ret_12m",
    "vol_21d", "vol_63d", "vol_ratio",
    "volume_zscore", "ret_vs_sector",
]

# ── 매크로 피처 15개 ──────────────────────────────────────────────────────────
MACRO_FEATURES = [
    "VIX_contango", "VIX_slope_9d_3m", "VIX_slope_3m_6m",
    "SKEW_level", "SKEW_zscore",
    "Cu_Au_ratio", "Cu_Au_ratio_chg",
    "HY_spread", "HY_spread_chg",
    "yield_curve", "yield_curve_inv",
    "claims_4wma", "claims_zscore",
    "WEI_level", "sahm_indicator",
]

# ── GDELT 감성 피처 ───────────────────────────────────────────────────────────
GDELT_FEATURES = [
    "gdelt_avg_tone_1m", "gdelt_tone_momentum", "gdelt_article_volume",
]

# ── HMM 레짐 확률 ─────────────────────────────────────────────────────────────
HMM_FEATURES = ["hmm_crisis_prob"]

# ── 수치형 전체 (sector_code 제외) ───────────────────────────────────────────
NUM_FEATURES = ASSET_FEATURES + MACRO_FEATURES + GDELT_FEATURES + HMM_FEATURES
ALL_FEATURES = NUM_FEATURES + [SECTOR_COL]

# ── 실제 존재하는 컬럼만 사용 (GDELT/HMM 수집 전 graceful 처리) ──────────────
available_num = [c for c in NUM_FEATURES if c in panel.columns]
available_cat = [SECTOR_COL] if SECTOR_COL in panel.columns else []
FINAL_FEATURES = available_num + available_cat

missing_cols = [c for c in ALL_FEATURES if c not in panel.columns]

print(f"수치형 피처: {len(available_num)}개")
print(f"카테고리 피처: {len(available_cat)}개  →  {available_cat}")
print(f"총 피처: {len(FINAL_FEATURES)}개")
if missing_cols:
    print(f"\n⚠️  아직 없는 피처 (수집 후 자동 추가됨): {missing_cols}")

## 4. Walk-Forward 날짜 윈도우 생성

**슬라이딩 방식**: IS 시작점을 OOS_DAYS씩 이동

```
Window 0:  IS[0:150]   →  Embargo[150:171]  →  OOS[171:201]
Window 1:  IS[30:180]  →  Embargo[180:201]  →  OOS[201:231]
...
```

> 💡 패널이 **월별** 주기라면 `IS_DAYS=7, EMBARGO_DAYS=1, OOS_DAYS=1` 로 재설정하세요.


In [ ]:
dates_arr = (panel.index
             .get_level_values("date")
             .unique()
             .sort_values()
             .to_numpy())
n_dates = len(dates_arr)

windows = []
i = 0
while True:
    oos_end = i + IS_DAYS + EMBARGO_DAYS + OOS_DAYS
    if oos_end > n_dates:
        break
    windows.append({
        "window_id"     : len(windows),
        "is_dates"      : dates_arr[i : i + IS_DAYS],
        "emb_dates"     : dates_arr[i + IS_DAYS : i + IS_DAYS + EMBARGO_DAYS],
        "oos_dates"     : dates_arr[i + IS_DAYS + EMBARGO_DAYS : oos_end],
        "oos_start_date": dates_arr[i + IS_DAYS + EMBARGO_DAYS],
        "oos_end_date"  : dates_arr[oos_end - 1],
    })
    i += OOS_DAYS  # 슬라이딩

print(f"총 날짜 수  : {n_dates}")
print(f"총 윈도우 수: {len(windows)}")
print("\n[윈도우 샘플]")
for w in windows[:3]:
    print(f"  W{w['window_id']:02d}: IS {w['is_dates'][0]!s:.10s}~{w['is_dates'][-1]!s:.10s} "
          f"| OOS {w['oos_start_date']!s:.10s}~{w['oos_end_date']!s:.10s}")
print("  ...")

## 5. 핵심 헬퍼 함수

In [ ]:
def discretize_labels(is_ret, oos_ret, n_classes=5):
    """
    IS 수익률 분위수로 bins 계산 → IS·OOS 레이블 반환
    ⚠️  bins는 IS 데이터만으로 결정 (look-ahead bias 완전 차단)

    Returns
    -------
    is_labels  : np.ndarray[int]  (0 ~ n_classes-1)
    oos_labels : np.ndarray[int]
    bins       : np.ndarray  분위수 경계값
    r_bar      : dict  {클래스 k → IS 구간 평균 수익률}
    """
    _, bins = pd.qcut(is_ret, q=n_classes, retbins=True, duplicates="drop")
    bins[0]  = -np.inf
    bins[-1] =  np.inf

    def _cut(arr):
        return (pd.cut(arr, bins=bins,
                       labels=list(range(n_classes)),
                       include_lowest=True)
                .astype(float).fillna(0).astype(int).values)

    is_labels  = _cut(is_ret)
    oos_labels = _cut(oos_ret)

    # IS 구간 클래스별 평균 수익률 r̄_k
    r_bar = {k: float(is_ret[is_labels == k].mean())
             if (is_labels == k).sum() > 0 else 0.0
             for k in range(n_classes)}

    return is_labels, oos_labels, bins, r_bar


def compute_Q_Omega(proba, r_bar, n_classes=5):
    """
    Q_i  = Σ_k  r̄_k · P(C_k | z_i)           ← 클래스 가중 기댓값
    Ω_ii = Σ_k  P(C_k | z_i) · (r̄_k - Q_i)²  ← 예측 분포 분산

    Parameters
    ----------
    proba  : (n_samples, n_classes)  XGBoost predict_proba 출력
    r_bar  : dict  {k → IS 클래스 k 평균 수익률}

    Returns
    -------
    Q     : (n_samples,)  기대수익률 벡터
    Omega : (n_samples,)  불확실성 (분산) 벡터
    """
    r_arr = np.array([r_bar[k] for k in range(n_classes)])  # (n_classes,)
    Q     = proba @ r_arr                                    # (n_samples,)
    Omega = np.sum(
        proba * (r_arr[np.newaxis, :] - Q[:, np.newaxis])**2,
        axis=1
    )
    return Q, Omega


def encode_sector(is_df, oos_df, col="sector_code"):
    """sector_code → 정수  (IS로 fit, OOS에 transform)"""
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    is_enc  = enc.fit_transform(is_df[[col]]).ravel().astype(int)
    oos_enc = enc.transform(oos_df[[col]]).ravel().astype(int)
    return is_enc, oos_enc


def r2_oos(y_true, y_pred):
    """OOS R² = 1 - SS_res/SS_tot  (양수이면 ML > Naive Mean)"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 1e-12 else np.nan


print("✅ 헬퍼 함수 정의 완료")

## 6. Optuna 하이퍼파라미터 탐색 함수

In [ ]:
def make_objective(X_tr, y_tr, n_classes, val_ratio=0.2):
    """
    Optuna objective 팩토리.
    IS를 80/20 시간 순 분할 → log_loss 최소화.
    (랜덤 분할 금지: 시계열 특성 보존)
    """
    split   = int(len(X_tr) * (1 - val_ratio))
    X_t, X_v = X_tr[:split], X_tr[split:]
    y_t, y_v = y_tr[:split], y_tr[split:]

    def objective(trial):
        params = dict(
            n_estimators      = trial.suggest_int("n_estimators", 100, 500),
            max_depth         = trial.suggest_int("max_depth", 3, 7),
            learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            subsample         = trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree  = trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_weight  = trial.suggest_int("min_child_weight", 1, 10),
            gamma             = trial.suggest_float("gamma", 0.0, 3.0),
            reg_lambda        = trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
            reg_alpha         = trial.suggest_float("reg_alpha", 1e-5, 1.0, log=True),
            # 고정값
            objective         = "multi:softprob",
            num_class         = n_classes,
            eval_metric       = "mlogloss",
            tree_method       = "hist",
            random_state      = RANDOM_STATE,
            n_jobs            = -1,
            verbosity         = 0,
        )
        model = XGBClassifier(**params)
        model.fit(X_t, y_t,
                  eval_set=[(X_v, y_v)],
                  early_stopping_rounds=20,
                  verbose=False)
        proba_v = model.predict_proba(X_v)
        return log_loss(y_v, proba_v)  # 최소화

    return objective


print("✅ Optuna objective 정의 완료")

## 7. Walk-Forward 메인 루프

각 윈도우마다:

1. **IS / OOS 분할** (엠바고 제외)
2. **5분위 이산화** → bins는 IS 기준, r̄_k 계산
3. **Optuna** 하이퍼파라미터 탐색 (IS 내부 80/20 시계열 분할)
4. **전체 IS**로 최종 XGBoost 학습
5. OOS 클래스 확률 예측 → **Q, Ω** 계산
6. Accuracy, R²_oos 기록


In [ ]:
all_results    = []   # OOS 행별 예측 결과
window_stats   = []   # 윈도우별 성과 요약
best_params_log = []  # 윈도우별 최적 하이퍼파라미터
final_model    = None # 마지막 윈도우 모델 (피처 중요도용)

dates_level_idx = panel.index.get_level_values("date")

for window in tqdm(windows, desc="Walk-Forward"):
    wid       = window["window_id"]
    is_set    = set(window["is_dates"].tolist())
    oos_set   = set(window["oos_dates"].tolist())

    # ── 1. IS / OOS 분할 ────────────────────────────────────────────────────
    is_df  = panel[dates_level_idx.isin(is_set)].copy()
    oos_df = panel[dates_level_idx.isin(oos_set)].copy()

    is_df  = is_df.dropna(subset=[TARGET_COL])
    oos_pred_df = oos_df.dropna(subset=available_num)   # 피처 완전한 행
    oos_eval_df = oos_df.dropna(subset=[TARGET_COL])    # 실제값 있는 행

    if len(is_df) < IS_DAYS // 2 or len(oos_pred_df) == 0:
        continue

    # ── 2. 5분위 이산화 (IS 기준 bins) ──────────────────────────────────────
    is_ret  = is_df[TARGET_COL].values
    # OOS 실제값 (평가용; 없으면 더미)
    oos_ret_eval = oos_eval_df[TARGET_COL].values if len(oos_eval_df) > 0 else None
    dummy_oos    = np.zeros(len(oos_pred_df))

    try:
        is_labels, _, bins, r_bar = discretize_labels(
            is_ret,
            dummy_oos,   # OOS bins 계산엔 IS 기준 적용
            n_classes=N_CLASSES
        )
    except Exception as e:
        print(f"  W{wid}: 이산화 실패 ({e}), 스킵")
        continue

    # ── 3. 피처 행렬 구성 ────────────────────────────────────────────────────
    X_is  = is_df[available_num].fillna(0).values
    X_oos = oos_pred_df[available_num].fillna(0).values
    y_is  = is_labels

    if available_cat:
        is_sec, oos_sec = encode_sector(is_df, oos_pred_df)
        X_is  = np.column_stack([X_is,  is_sec])
        X_oos = np.column_stack([X_oos, oos_sec])

    # ── 4. Optuna 탐색 ───────────────────────────────────────────────────────
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE + wid),
    )
    study.optimize(
        make_objective(X_is, y_is, N_CLASSES),
        n_trials=N_OPTUNA_TRIALS,
        show_progress_bar=False,
    )

    best_p = study.best_params.copy()
    best_p.update({
        "objective"   : "multi:softprob",
        "num_class"   : N_CLASSES,
        "eval_metric" : "mlogloss",
        "tree_method" : "hist",
        "random_state": RANDOM_STATE,
        "n_jobs"      : -1,
        "verbosity"   : 0,
    })
    best_params_log.append({"window_id": wid, **best_p})

    # ── 5. 전체 IS로 최종 학습 ───────────────────────────────────────────────
    final_model = XGBClassifier(**best_p)
    final_model.fit(X_is, y_is, verbose=False)

    # ── 6. OOS 확률 예측 → Q, Ω ─────────────────────────────────────────────
    proba_oos = final_model.predict_proba(X_oos)   # (n_oos, 5)
    Q_arr, Omega_arr = compute_Q_Omega(proba_oos, r_bar, N_CLASSES)

    # ── 7. 평가 (실제값 있는 행에서만) ──────────────────────────────────────
    acc = r2 = np.nan
    if oos_ret_eval is not None and len(oos_eval_df) > 0:
        pred_idx = oos_pred_df.index
        eval_idx = oos_eval_df.index
        common   = pred_idx.intersection(eval_idx)
        if len(common) > 0:
            # 정렬 인덱스 매핑
            pred_pos = [list(pred_idx).index(i) for i in common]
            eval_pos = [list(eval_idx).index(i) for i in common]

            _, oos_labels_common, _, _ = discretize_labels(
                is_ret,
                oos_eval_df.iloc[eval_pos][TARGET_COL].values,
                n_classes=N_CLASSES
            )
            pred_classes = final_model.predict(X_oos[pred_pos])
            acc = accuracy_score(oos_labels_common, pred_classes)
            r2  = r2_oos(oos_eval_df.iloc[eval_pos][TARGET_COL].values,
                         Q_arr[pred_pos])

    # ── 8. 행별 결과 저장 ────────────────────────────────────────────────────
    eval_ret_map = {}
    if oos_ret_eval is not None:
        for i2, idx2 in enumerate(oos_eval_df.index):
            eval_ret_map[idx2] = oos_eval_df.iloc[i2][TARGET_COL]

    for i, idx in enumerate(oos_pred_df.index):
        row = {
            "date"     : idx[0],
            "ticker"   : idx[1],
            "Q"        : Q_arr[i],
            "Omega"    : Omega_arr[i],
            "actual_ret": eval_ret_map.get(idx, np.nan),
            "window_id": wid,
        }
        for c in range(N_CLASSES):
            row[f"proba_C{c+1}"] = float(proba_oos[i, c])
        all_results.append(row)

    window_stats.append({
        "window_id"      : wid,
        "oos_start"      : window["oos_start_date"],
        "oos_end"        : window["oos_end_date"],
        "is_size"        : len(is_df),
        "oos_size"       : len(oos_pred_df),
        "oos_accuracy"   : acc,
        "oos_r2"         : r2,
        "optuna_best_logloss": study.best_value,
        "r_bar_C1"       : r_bar[0],
        "r_bar_C5"       : r_bar[4],
    })

print(f"\n✅ Walk-Forward 완료: {len(window_stats)}개 윈도우 처리")

## 8. 결과 집계 및 저장

In [ ]:
results_df = (pd.DataFrame(all_results)
              .set_index(["date", "ticker"])
              .sort_index())
stats_df   = pd.DataFrame(window_stats)
params_df  = pd.DataFrame(best_params_log)

results_df.to_parquet(OUTPUT_DIR / "xgb_Q_Omega.parquet")
stats_df.to_csv(OUTPUT_DIR / "xgb_window_stats.csv", index=False)
params_df.to_csv(OUTPUT_DIR / "xgb_best_params_log.csv", index=False)

print(f"xgb_Q_Omega.parquet     → shape {results_df.shape}")
print(f"xgb_window_stats.csv    → shape {stats_df.shape}")
print(f"xgb_best_params_log.csv → shape {params_df.shape}")

results_df.head(6)

## 9. 성과 요약 통계

In [ ]:
print("=" * 55)
print("  XGBoost Walk-Forward OOS 성과 요약")
print("=" * 55)

acc_s = stats_df["oos_accuracy"].dropna()
r2_s  = stats_df["oos_r2"].dropna()

print(f"\n[5분위 분류 Accuracy]")
print(f"  평균   : {acc_s.mean():.4f}  (랜덤 기준선: {1/N_CLASSES:.4f})")
print(f"  중앙값 : {acc_s.median():.4f}")
print(f"  std    : {acc_s.std():.4f}")
print(f"  범위   : {acc_s.min():.4f} ~ {acc_s.max():.4f}")

print(f"\n[R²_oos  (Q vs 실제수익률)]")
print(f"  평균         : {r2_s.mean():.4f}")
print(f"  R²>0 비율    : {(r2_s>0).mean():.1%}  ← ML > Naive Mean인 윈도우 비율")
print(f"  범위         : {r2_s.min():.4f} ~ {r2_s.max():.4f}")

print(f"\n[Q / Ω 분포]")
print(f"  Q   평균: {results_df['Q'].mean():.5f}  std: {results_df['Q'].std():.5f}")
print(f"  Ω   평균: {results_df['Omega'].mean():.6f}  std: {results_df['Omega'].std():.6f}")

## 10. 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("XGBoost Walk-Forward 성과 분석", fontsize=14, fontweight="bold")

# (1) OOS Accuracy by Window
ax = axes[0, 0]
ax.plot(stats_df["window_id"], stats_df["oos_accuracy"],
        marker="o", ms=3, lw=1.2, color="steelblue", label="Accuracy")
ax.axhline(1 / N_CLASSES, color="tomato", ls="--", lw=1.5,
           label=f"Random ({1/N_CLASSES:.2f})")
ax.set_title("OOS 5분위 분류 Accuracy"); ax.set_xlabel("Window ID")
ax.set_ylabel("Accuracy"); ax.legend(); ax.grid(alpha=0.3)

# (2) R²_oos by Window
ax = axes[0, 1]
colors_bar = ["limegreen" if v > 0 else "tomato"
               for v in stats_df["oos_r2"].fillna(0)]
ax.bar(stats_df["window_id"], stats_df["oos_r2"].fillna(0),
       color=colors_bar, alpha=0.75)
ax.axhline(0, color="black", lw=1.2)
ax.set_title("R²_oos  (녹색: ML > Naive Mean)"); ax.set_xlabel("Window ID")
ax.set_ylabel("R²_oos"); ax.grid(alpha=0.3, axis="y")

# (3) Q 분포
ax = axes[1, 0]
ax.hist(results_df["Q"], bins=60, color="steelblue", alpha=0.75, edgecolor="white")
ax.axvline(0, color="red", ls="--", lw=1.2)
ax.set_title("Q (예측 기대수익률) 분포")
ax.set_xlabel("Q"); ax.set_ylabel("빈도"); ax.grid(alpha=0.3)

# (4) Ω 분포
ax = axes[1, 1]
ax.hist(results_df["Omega"], bins=60, color="darkorange", alpha=0.75, edgecolor="white")
ax.set_title("Ω (예측 불확실성) 분포")
ax.set_xlabel("Ω"); ax.set_ylabel("빈도"); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xgb_walkforward_performance.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR / 'xgb_walkforward_performance.png'}")

## 11. Q 히트맵 (ETF × 날짜)

In [ ]:
q_pivot = results_df["Q"].unstack("ticker")

n_hm = len(q_pivot)
step = max(1, n_hm // 20)
xlabels = [str(d)[:7] if i % step == 0 else ""
           for i, d in enumerate(q_pivot.index)]

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(q_pivot.T, cmap="RdYlGn", center=0, ax=ax,
            xticklabels=xlabels, yticklabels=True,
            cbar_kws={"label": "Q (기대수익률)"})
ax.set_title("Q 히트맵 — ETF × 날짜  (빨강=하락전망 / 녹색=상승전망)", fontsize=12)
ax.set_xlabel("날짜"); ax.set_ylabel("티커")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xgb_Q_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
omega_pivot = results_df["Omega"].unstack("ticker")

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(omega_pivot.T, cmap="YlOrRd", ax=ax,
            xticklabels=xlabels, yticklabels=True,
            cbar_kws={"label": "Ω (불확실성)"})
ax.set_title("Ω 히트맵 — ETF × 날짜  (진할수록 불확실성 높음)", fontsize=12)
ax.set_xlabel("날짜"); ax.set_ylabel("티커")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "xgb_Omega_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. 피처 중요도 (마지막 윈도우 기준)

In [ ]:
if final_model is not None:
    feat_names = available_num + available_cat
    importance = (pd.Series(final_model.feature_importances_, index=feat_names)
                  .sort_values(ascending=True))

    fig, ax = plt.subplots(figsize=(9, max(6, len(feat_names) * 0.28)))
    importance.plot(kind="barh", ax=ax, color="steelblue", alpha=0.8)
    ax.set_title(f"XGBoost 피처 중요도 (마지막 윈도우)", fontsize=12)
    ax.set_xlabel("Importance (gain)"); ax.grid(alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "xgb_feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\n[상위 10 피처]")
    print(importance.sort_values(ascending=False).head(10).to_string())

## 13. BL 입력용 Q, Ω 최종 저장

`Step 3-2 Black-Litterman` 노트북에서 바로 사용할 수 있도록
**(date × ticker)** 피벗 형태로 저장한다.


In [ ]:
Q_bl     = results_df["Q"].unstack("ticker")
Omega_bl = results_df["Omega"].unstack("ticker")

Q_bl.to_parquet(OUTPUT_DIR / "Q_xgb.parquet")
Omega_bl.to_parquet(OUTPUT_DIR / "Omega_xgb.parquet")

print("=" * 50)
print("✅ BL 입력 파일 저장 완료")
print(f"  Q_xgb.parquet     shape: {Q_bl.shape}")
print(f"  Omega_xgb.parquet shape: {Omega_bl.shape}")
print(f"  저장 위치: {OUTPUT_DIR.resolve()}")
print("\n[Q 마지막 5 날짜 샘플]")
Q_bl.tail()

In [ ]:
Omega_bl.tail()

## 14. 다음 단계

| 단계 | 작업 | 파일 |
|------|------|------|
| Step 3-1 (B) | Random Forest → Q_rf, Ω_rf | `Step3_RF_WalkForward.ipynb` |
| Step 3-1 (C) | LLM Agent → Q_llm, Ω_llm | `Step3_LLM_Agent.ipynb` |
| Step 3-2 | 모델별 성과 비교 + Diebold-Mariano 검정 → 최적 Q, Ω 선택 | `Step3_Model_Selection.ipynb` |
| Step 3-3 | Black-Litterman → μ_BL, Σ_BL | `Step3_BlackLitterman.ipynb` |
| Step 3-4 | MVO (γ별) → 성향별 최적 비중 w* | `Step3_MVO.ipynb` |

---

> **모델 선택 기준 (Step 3-2)**
> RF / XGBoost / TabPFN 각각의 OOS Accuracy, R²_oos, Sharpe(τ* 그리드서치)를
> 독립적으로 비교한 뒤, 가장 우수한 단일 모델의 Q, Ω를 BL에 투입한다.
> (무조건 앙상블 시 신호 상쇄 → 시장 추종으로 수렴 위험)
